# Stage 10 — Ship the current model as a shadow/demo release

**Goal:** Assemble the verified model, API code, tests, and release evidence into one portable ZIP.

**Why:** A shippable model needs more than `model.joblib`: it needs an input contract, environment versions, an integrity check, test evidence, and a clear operational status. This release remains **HOLD_FOR_OPERATIONAL_USE** because HEAT/HOT WATER recall was zero in the final test.

Download `NYC311_Shadow_Release_Source.zip` from the teacher's message before Cell 2. Cell 2 asks you to upload that source ZIP into Colab. The trained model stays in your Drive until the verified assembly step.

In [ ]:
# Cell 2 — Why: accept only the exact tested source code.
from google.colab import files
from pathlib import Path
from datetime import datetime, timezone
import hashlib, io, zipfile

source_name = "NYC311_Shadow_Release_Source.zip"
source_sha256 = "718f44215a5cb9b855858aab56100cc5ed508bccaca2ab1f9d6fe4f97e1ef81a"
uploaded = files.upload()
if set(uploaded) != {source_name}:
    raise ValueError(f"Upload exactly {source_name}")
source_bytes = uploaded[source_name]
if hashlib.sha256(source_bytes).hexdigest() != source_sha256:
    raise ValueError("Source ZIP changed")

build_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
release_dir = Path("/content") / f"nyc311_shadow_release_{build_id}"
release_dir.mkdir(parents=True, exist_ok=False)
with zipfile.ZipFile(io.BytesIO(source_bytes)) as archive:
    archive.extractall(release_dir)
print("Verified source files:", len(list(release_dir.rglob("*.py"))))

In [ ]:
# Cell 3 — Why: copy the exact model and evidence approved for this shadow demo.
from google.colab import drive
import json, shutil

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/nyc311_mlops")
final_id = "20260926T143549799163Z"
decision_id = "20260926T143947829974Z"
experiment_id = "20260926T143147282446Z"
split_id = "20260926T142339294958Z"

final_dir = root / "final_evaluations" / final_id
decision_file = root / "release_decisions" / decision_id / "decision.json"
plan_file = final_dir / "locked_plan.json"
report_file = final_dir / "test_report.json"
experiment_file = root / "experiments" / experiment_id / "experiment.json"
decision = json.loads(decision_file.read_text())
plan = json.loads(plan_file.read_text())
report = json.loads(report_file.read_text())

if decision["decision"] != "HOLD_FOR_OPERATIONAL_USE":
    raise ValueError("Wrong release status")
if (decision["final_evaluation_id"] != final_id or
        decision["experiment_id"] != experiment_id or
        decision["split_id"] != split_id):
    raise ValueError("Wrong release lineage")
for file, key in [(plan_file, "locked_plan_sha256"),
                  (report_file, "test_report_sha256"),
                  (experiment_file, "experiment_sha256")]:
    if hashlib.sha256(file.read_bytes()).hexdigest() != decision[key]:
        raise ValueError(f"Changed evidence: {file.name}")
if plan["model_sha256"] != hashlib.sha256((final_dir / "model.joblib").read_bytes()).hexdigest():
    raise ValueError("Final model changed")
if report["final_id"] != final_id or report["groups"]["HEAT/HOT WATER"]["recall"] != 0.0:
    raise ValueError("Wrong test report")

shutil.copy2(final_dir / "model.joblib", release_dir / "model.joblib")
evidence = release_dir / "evidence"
evidence.mkdir(exist_ok=True)
for file in [decision_file, plan_file, report_file, experiment_file]:
    shutil.copy2(file, evidence / file.name)
print("Verified model:", final_id)
print("Release decision:", decision["decision"])

In [ ]:
# Cell 4 — Why: pin the packages used to load and serve the model.
import importlib.metadata as package_metadata
import platform, subprocess, sys

for package in ["fastapi", "uvicorn", "httpx"]:
    try:
        package_metadata.version(package)
    except package_metadata.PackageNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", package], check=True)

versions = plan["versions"]
if platform.python_version() != versions["python"]:
    raise RuntimeError("Use the same Python version as the final fit")
for package, key in [("pandas", "pandas"), ("numpy", "numpy"),
                     ("scikit-learn", "scikit_learn"), ("joblib", "joblib")]:
    if package_metadata.version(package) != versions[key]:
        raise RuntimeError(f"{package} version differs from final fit")

pins = ["pandas", "numpy", "scikit-learn", "joblib",
        "fastapi", "pydantic", "starlette", "uvicorn", "httpx"]
requirements = [f"{name}=={package_metadata.version(name)}" for name in pins]
(release_dir / "requirements.txt").write_text("\n".join(requirements) + "\n")
print("Pinned packages:", requirements)

In [ ]:
# Cell 5 — Why: save release metadata and run the real model smoke check.
from datetime import datetime, timezone

release_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
release = {
    "release_id": release_id,
    "mode": "shadow",
    "operational_status": "HOLD_FOR_OPERATIONAL_USE",
    "final_evaluation_id": final_id,
    "decision_id": decision_id,
    "experiment_id": experiment_id,
    "split_id": split_id,
    "model_sha256": plan["model_sha256"],
    "source_sha256": source_sha256,
    "threshold": 0.2,
    "features": plan["features"],
    "versions": versions,
    "test_overall": report["overall"],
    "heat_hot_water_test": report["groups"]["HEAT/HOT WATER"],
}
(release_dir / "release.json").write_text(json.dumps(release, indent=2) + "\n")

workflow_dir = release_dir / ".github" / "workflows"
workflow_dir.mkdir(parents=True, exist_ok=True)
python_minor = ".".join(versions["python"].split(".")[:2])
workflow = '''name: Shadow release checks
on: [push, pull_request, workflow_dispatch]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: PYTHON_VERSION
      - run: python -m pip install -r requirements.txt
      - run: python -m unittest discover -s tests -v
      - run: python -c "from nyc311_shadow.core import load_release; load_release('.')"
'''.replace("PYTHON_VERSION", python_minor)
(workflow_dir / "checks.yml").write_text(workflow)

subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
               cwd=release_dir, check=True)
sys.path.insert(0, str(release_dir))
from nyc311_shadow.core import load_release

service = load_release(release_dir)
example = {"complaint_type": "HEAT/HOT WATER", "borough": "BRONX",
           "created_date": "2026-09-26T10:00:00"}
print("Smoke prediction:", service.predict(example))
import os
from fastapi.testclient import TestClient
from nyc311_shadow.api import app

os.environ["NYC311_RELEASE_DIR"] = str(release_dir)
with TestClient(app) as client:
    health = client.get("/health")
    prediction = client.post("/predict", json=example)
    rejected = client.post("/predict", json={**example, "closed_date": "2026-09-27T10:00:00"})
assert health.status_code == 200 and health.json()["operational_status"] == "HOLD_FOR_OPERATIONAL_USE"
assert prediction.status_code == 200 and prediction.json()["mode"] == "shadow"
assert rejected.status_code == 422
print("API checks: /health 200, /predict 200, leaked input 422")
print("Release ID:", release_id)

In [ ]:
# Cell 6 — Why: create a portable, verified release ZIP in Drive.
import shutil

bundle_dir = root / "release_bundles"
bundle_dir.mkdir(parents=True, exist_ok=True)
bundle = Path(shutil.make_archive(str(bundle_dir / release_id), "zip", root_dir=release_dir))
bundle_sha256 = hashlib.sha256(bundle.read_bytes()).hexdigest()
print("Release ZIP:", bundle)
print("Release ZIP SHA-256:", bundle_sha256)
print("Mode: shadow; operational status: HOLD_FOR_OPERATIONAL_USE")
files.download(str(bundle))
print("A copy remains in Drive")

# Stage 10 takeaway and quiz

**What you learned:** A release bundle includes the model, code, dependencies, checksums, evidence, and a clear decision. The local API can now be run from the bundle. No external service has been published by this notebook.

**Easy quiz:**

1. Why do we verify the model's SHA-256 before loading it?
2. Why does `/health` include both the model ID and HOLD status?
3. Why do we keep the training data outside the release ZIP?
4. What does shadow mode prevent us from doing?

Send me the Release ID, Smoke prediction, ZIP path, and ZIP SHA-256. The next stage puts the source in a GitHub repository and runs its checks automatically.